# Exploratory Data Analysis — Stroke Digital Twin

Analysis of the MIMIC-IV stroke cohort extracted for the digital twin project.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

sns.set_theme(style='whitegrid', font_scale=1.2)

with open('../config/config.yaml') as f:
    config = yaml.safe_load(f)

## 1. Load Cohort Data

In [ ]:
cohort = pd.read_parquet('../outputs/cohort/stroke_cohort.parquet')
static = pd.read_parquet('../outputs/cohort/static_features.parquet')
ts = pd.read_parquet('../outputs/cohort/timeseries_processed.parquet')

print(f'Cohort size: {len(cohort):,} patients')
print(f'Static features: {static.shape[1]} columns')
print(f'Time-series rows: {len(ts):,}')

## 2. Demographics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age distribution
axes[0].hist(static['anchor_age'], bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].set_title('Age Distribution')
axes[0].axvline(static['anchor_age'].median(), color='red', linestyle='--', label=f"Median: {static['anchor_age'].median():.0f}")
axes[0].legend()

# Gender
static['gender'].value_counts().plot.bar(ax=axes[1], edgecolor='black', alpha=0.7)
axes[1].set_title('Gender Distribution')
axes[1].set_ylabel('Count')

# Mortality
static['hospital_expire_flag'].value_counts().plot.bar(ax=axes[2], edgecolor='black', alpha=0.7)
axes[2].set_title('In-Hospital Mortality')
axes[2].set_xticklabels(['Survived', 'Died'], rotation=0)
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../outputs/figures/demographics.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Stroke Subtypes

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
subtype_counts = static['stroke_subtype'].value_counts()
subtype_counts.plot.bar(ax=ax, edgecolor='black', alpha=0.7, color=sns.color_palette('Set2'))
ax.set_title('Stroke Subtype Distribution')
ax.set_ylabel('Count')
ax.set_xlabel('')
for i, (idx, val) in enumerate(subtype_counts.items()):
    ax.text(i, val + 20, f'{val} ({val/len(static)*100:.1f}%)', ha='center', fontsize=10)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../outputs/figures/stroke_subtypes.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Comorbidity Prevalence

In [ ]:
comorbidity_cols = ['has_hypertension', 'has_diabetes', 'has_afib', 
                    'has_dyslipidemia', 'has_ckd', 'has_cad']
prevalence = static[comorbidity_cols].mean().sort_values(ascending=True) * 100

fig, ax = plt.subplots(figsize=(8, 5))
prevalence.plot.barh(ax=ax, edgecolor='black', alpha=0.7, color=sns.color_palette('Blues_r', len(prevalence)))
ax.set_xlabel('Prevalence (%)')
ax.set_title('Comorbidity Prevalence in Stroke Cohort')
for i, v in enumerate(prevalence):
    ax.text(v + 0.5, i, f'{v:.1f}%', va='center')
plt.tight_layout()
plt.savefig('../outputs/figures/comorbidities.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. ICU Time-Series Overview

In [ ]:
# Missing data rates
vital_cols = ['hr', 'sbp', 'dbp', 'map', 'rr', 'spo2', 'temp_c', 
              'gcs_eye', 'gcs_verbal', 'gcs_motor', 'gcs_total']
existing = [c for c in vital_cols if c in ts.columns]
missing_rates = ts[existing].isna().mean() * 100

fig, ax = plt.subplots(figsize=(10, 5))
missing_rates.sort_values().plot.barh(ax=ax, edgecolor='black', alpha=0.7)
ax.set_xlabel('Missing Rate (%)')
ax.set_title('Time-Series Missing Data Rates (after forward-fill)')
plt.tight_layout()
plt.savefig('../outputs/figures/ts_missing_rates.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sample patient trajectories (3 random patients)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sample_stays = ts['stay_id'].unique()[:3]

for stay_id in sample_stays:
    pt = ts[ts['stay_id'] == stay_id].sort_values('hour')
    if 'hr' in pt.columns:
        axes[0, 0].plot(pt['hour'], pt['hr'], alpha=0.7, label=f'Stay {stay_id}')
    if 'spo2' in pt.columns:
        axes[0, 1].plot(pt['hour'], pt['spo2'], alpha=0.7)
    if 'sbp' in pt.columns:
        axes[1, 0].plot(pt['hour'], pt['sbp'], alpha=0.7)
    if 'gcs_total' in pt.columns:
        axes[1, 1].plot(pt['hour'], pt['gcs_total'], alpha=0.7)

axes[0, 0].set_title('Heart Rate'); axes[0, 0].set_ylabel('bpm')
axes[0, 1].set_title('SpO2'); axes[0, 1].set_ylabel('%')
axes[1, 0].set_title('Systolic BP'); axes[1, 0].set_ylabel('mmHg')
axes[1, 1].set_title('GCS Total'); axes[1, 1].set_ylabel('Score')
for ax in axes.flat:
    ax.set_xlabel('Hours since ICU admission')
plt.suptitle('Sample Patient ICU Trajectories', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/figures/sample_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Admission Lab Distributions

In [ ]:
lab_cols = [c for c in static.columns if c.endswith('_admit')]
if lab_cols:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for i, col in enumerate(lab_cols[:6]):
        ax = axes[i // 3, i % 3]
        static[col].dropna().hist(bins=30, ax=ax, edgecolor='black', alpha=0.7)
        ax.set_title(col.replace('_admit', '').title())
        ax.set_ylabel('Count')
    plt.suptitle('Admission Lab Distributions', fontsize=14)
    plt.tight_layout()
    plt.savefig('../outputs/figures/admission_labs.png', dpi=150, bbox_inches='tight')
    plt.show()